<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> Fine-Tuning GPT-OSS 20B for Tool Use</h1>

Fine-tune GPT-OSS 20B to use tools reliably with **Unsloth QLoRA** (~14GB VRAM).

## What You'll Learn

1. Generate training data that teaches tool-calling behavior
2. Load a 20B model in 4-bit quantization for memory-efficient training
3. Apply LoRA adapters and train with loss masking
4. Test and save the fine-tuned model

## Prerequisites

- GPU with 16GB+ VRAM
- `tk-jupyter-fine-tuning` image
- Model `unsloth/gpt-oss-20b-bnb-4bit` mirrored to MLflow

---
## 1. Setup

In [ ]:
import os
import json
import random
import torch

from check_jupyter_flavor import check_flavor
check_flavor('fine-tuning')

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI')
print(f"MLflow: {MLFLOW_TRACKING_URI}")

max_seq_length = 1024

---
## 2. Generate Training Data

Create examples that teach the model to call tools and **stop** (not hallucinate results).

```
User: Find papers about transformers
Assistant: search_papers({"query": "transformers"})
[END - model learns to stop here]
```

At inference, an orchestration system (CrewAI, LangChain) executes the real tool.

In [ ]:
def generate_training_examples():
    """Generate tool-call training examples."""
    examples = []
    
    # search_papers examples
    search_queries = [
        ("Find papers about transformer architectures", {"query": "transformer architectures"}),
        ("Search for research on attention mechanisms", {"query": "attention mechanisms"}),
        ("What papers exist about neural scaling laws?", {"query": "neural scaling laws"}),
        ("Look up papers on gradient optimization", {"query": "gradient optimization"}),
        ("I need papers on regularization techniques", {"query": "regularization techniques"}),
        ("Can you find research about deep learning?", {"query": "deep learning"}),
        ("Search the database for BERT models", {"query": "BERT models"}),
        ("What's the latest research on large language models?", {"query": "large language models"}),
        ("Find papers about protein folding prediction", {"query": "protein folding prediction"}),
        ("Search for research on climate modeling", {"query": "climate modeling"}),
        ("Look up papers on quantum error correction", {"query": "quantum error correction"}),
        ("Find papers about object detection", {"query": "object detection"}),
        ("Search for research on image segmentation", {"query": "image segmentation"}),
        ("What papers exist about reinforcement learning?", {"query": "reinforcement learning"}),
        ("Find academic papers on diffusion models", {"query": "diffusion models"}),
        ("Show me papers on federated learning", {"query": "federated learning"}),
        ("Get me research about GANs", {"query": "generative adversarial networks"}),
        ("Search for knowledge graph papers", {"query": "knowledge graphs"}),
        ("Find papers about motion planning", {"query": "motion planning"}),
        ("Look up SLAM algorithm research", {"query": "SLAM algorithms"}),
    ]
    for query, args in search_queries:
        examples.append({"user": query, "tool": "search_papers", "args": args})
    
    # get_paper_details examples
    details_queries = [
        ("Get me details of the Attention Is All You Need paper", {"paper_title": "Attention Is All You Need"}),
        ("Show me details about the BERT paper", {"paper_title": "BERT"}),
        ("I want to read more about the GPT-3 paper", {"paper_title": "GPT-3"}),
        ("Get details on the AlphaFold paper", {"paper_title": "AlphaFold"}),
        ("Show me the ResNet paper", {"paper_title": "ResNet"}),
    ]
    for query, args in details_queries:
        examples.append({"user": query, "tool": "get_paper_details", "args": args})
    
    # log_to_mlflow examples
    mlflow_queries = [
        ("Log my transformer analysis to MLflow", 
         {"experiment": "transformer-research", "summary": "Analysis complete"}),
        ("Save my findings to MLflow",
         {"experiment": "research-findings", "summary": "Research findings"}),
        ("Record my experiment results in MLflow",
         {"experiment": "experiment-results", "summary": "Experiment complete"}),
    ]
    for query, args in mlflow_queries:
        examples.append({"user": query, "tool": "log_to_mlflow", "args": args})
    
    return examples


def to_text_format(example):
    """Convert to training text format."""
    return f"""User: {example["user"]}
Assistant: {example["tool"]}({json.dumps(example["args"])})"""


# Generate and expand dataset
base_examples = generate_training_examples()
print(f"Base examples: {len(base_examples)}")

# Repeat to get ~500 examples
training_texts = []
while len(training_texts) < 500:
    for ex in base_examples:
        if len(training_texts) >= 500:
            break
        training_texts.append({"text": to_text_format(ex)})

random.shuffle(training_texts)

print(f"Training examples: {len(training_texts)}")
print(f"\nSample:")
print(training_texts[0]["text"])

In [ ]:
from datasets import Dataset

raw_dataset = Dataset.from_list(training_texts)
print(f"Dataset: {len(raw_dataset)} examples")

---
## 3. Load Model with Unsloth

Load GPT-OSS 20B in 4-bit quantization from MLflow.

In [ ]:
import requests
import urllib3
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def get_model_path(model_id: str) -> Path:
    """Get model path from MLflow."""
    model_name = model_id.replace('/', '-')
    mlflow_url = os.environ.get('MLFLOW_TRACKING_URI')
    token_url = os.environ.get('MLFLOW_KEYCLOAK_TOKEN_URL')
    
    # Get auth token
    token = None
    if token_url:
        resp = requests.post(token_url, data={
            'grant_type': 'password',
            'client_id': os.environ.get('MLFLOW_KEYCLOAK_CLIENT_ID', 'mlflow'),
            'client_secret': os.environ.get('MLFLOW_CLIENT_SECRET'),
            'username': os.environ.get('MLFLOW_AUTH_USERNAME'),
            'password': os.environ.get('MLFLOW_AUTH_PASSWORD'),
            'scope': 'openid'
        }, verify=False, timeout=30)
        resp.raise_for_status()
        token = resp.json()['access_token']
    
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    
    # Query model versions
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/model-versions/search",
        params={'filter': f"name='{model_name}'"},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    
    versions = resp.json().get('model_versions', [])
    if not versions:
        raise ValueError(f"Model '{model_name}' not found in MLflow")
    
    latest = max(versions, key=lambda v: int(v['version']))
    run_id = latest['run_id']
    
    # Get experiment ID
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/runs/get",
        params={'run_id': run_id},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    experiment_id = resp.json()['run']['info']['experiment_id']
    
    # Find path on JuiceFS
    for base in [Path('/home/jovyan/thinkube/mlflow'), Path.home() / 'thinkube' / 'mlflow']:
        path = base / 'artifacts' / experiment_id / run_id / 'artifacts' / 'model'
        if path.exists():
            return path
    
    raise FileNotFoundError(f"Model not found on filesystem")


# Get model path
MODEL_ID = "unsloth/gpt-oss-20b-bnb-4bit"
model_path = get_model_path(MODEL_ID)
print(f"Model path: {model_path}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(model_path),
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
    device_map={"": 0},
)

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

---
## 4. Format Data for Training

Convert to GPT-OSS Harmony format using the tokenizer's chat template.

In [ ]:
def format_for_harmony(examples):
    """Convert to Harmony chat format."""
    texts = []
    for text in examples["text"]:
        lines = text.strip().split("\n")
        user_msg = lines[0].replace("User: ", "").strip()
        assistant_msg = lines[1].replace("Assistant: ", "").strip()
        
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
        
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        # Ensure channel marker is present
        formatted = formatted.replace(
            "<|start|>assistant<|message|>",
            "<|start|>assistant<|channel|>final<|message|>"
        )
        texts.append(formatted)
    
    return {"text": texts}


train_dataset = raw_dataset.map(format_for_harmony, batched=True, batch_size=50)
train_dataset = train_dataset.filter(lambda x: len(x["text"]) > 0)

print(f"Formatted dataset: {len(train_dataset)} examples")
print(f"\nSample:")
print(train_dataset[0]["text"])

---
## 5. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.3f}%)")

---
## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

# Only compute loss on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>"
)

print("Starting training...")
stats = trainer.train()
print(f"\nDone! Final loss: {stats.training_loss:.4f}")

---
## 7. Test the Model

In [ ]:
import re

FastLanguageModel.for_inference(model)

test_prompts = [
    "Find papers about transformer architectures",
    "Search for research on protein folding",
    "Log my analysis to MLflow",
]

print("Testing tool-use capability:\n")

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    formatted = formatted.replace("<|start|>assistant<|message|>", "<|start|>assistant<|channel|>final<|message|>")
    
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # Extract tool call
    match = re.search(r'(\w+)\s*\(\s*(\{[^}]+\})\s*\)', response)
    if match:
        print(f"[USER] {prompt}")
        print(f"[TOOL] {match.group(1)}({match.group(2)})")
    else:
        print(f"[USER] {prompt}")
        print(f"[RAW] {response[:150]}...")
    print("-" * 60)

---
## 8. Save the Model

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/home/jovyan/thinkube/models/gpt-oss-tool-use")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapters
LORA_DIR = OUTPUT_DIR / "lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA saved: {LORA_DIR}")

# Save merged model
MERGED_DIR = OUTPUT_DIR / "merged"
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Merged saved: {MERGED_DIR}")

---
## Summary

| Step | What Happened |
|------|---------------|
| Generate Data | Created tool-call examples (no fake results) |
| Load Model | GPT-OSS 20B in 4-bit (~14GB VRAM) |
| Add LoRA | Trainable adapters (r=8, alpha=16) |
| Train | SFT with loss masking on responses only |
| Test | Model generates tool calls and stops |
| Save | LoRA adapters + merged model |

The fine-tuned model can now be deployed with CrewAI, LangChain, or other orchestration systems that handle real tool execution.